# EDA Goodreads - Fantasy & Paranormal

Este notebook explora muestras reproducibles de libros e interacciones. El objetivo es entender calidad, nulos, duplicados, distribución de ratings, temporalidad y relaciones entre metadatos antes de ejecutar la curación completa en `src/`.

In [ ]:
from pathlib import Path
import os, sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.config import CATEGORIES
from src.utils.io import read_jsonl_sample
from src.utils.cleaning import clean_books, clean_interactions
from src.utils.eda import missing_summary, duplicate_summary, iqr_outlier_summary

cfg = CATEGORIES['fantasy_paranormal']
sns.set_theme(style='whitegrid')

## Carga de muestra

Los archivos de interacciones son grandes y comprimidos con gzip, por lo que el EDA usa muestra para iterar rápido. El pipeline de curación procesa el archivo completo por chunks.

In [ ]:
books_raw = read_jsonl_sample(cfg.books_file, nrows=50_000)
interactions_raw = read_jsonl_sample(cfg.interactions_file, nrows=250_000)
books = clean_books(books_raw)
interactions = clean_interactions(interactions_raw)
books.shape, interactions.shape

## Esquema, nulos y strings vacíos

Se revisan tipos, nulos reales y strings vacíos. Goodreads usa strings vacíos para muchos campos opcionales; por eso la limpieza los convierte a nulos antes de castear tipos.

In [ ]:
display(missing_summary(books_raw).head(20))
display(missing_summary(interactions_raw).head(20))

## Ratings

`average_rating` describe libros; `rating` describe interacciones de usuarios. En las interacciones, `rating = 0` representa ausencia de rating explícito y se modela como `rating_missing=True` con `rating_clean` nulo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(books['average_rating'].dropna(), bins=30, ax=axes[0])
axes[0].set_title('Average rating por libro')
sns.countplot(data=interactions, x='rating', ax=axes[1])
axes[1].set_title('Rating de usuario, incluyendo 0 = sin rating')
plt.tight_layout()
interactions[['rating', 'rating_clean', 'rating_missing']].head()

## Publicaciones en el tiempo

La fecha se arma desde año, mes y día cuando existen. Si mes o día faltan, se usa el primer mes/día solo para una fecha ordenable; el año limpio se conserva como feature independiente.

In [ ]:
year_counts = books['publication_year'].dropna().astype(int).value_counts().sort_index()
year_counts.loc[year_counts.index.between(1800, 2025)].plot(figsize=(12, 4), title='Libros por año de publicación')
plt.xlabel('Año')
plt.ylabel('Cantidad')

## Correlaciones y outliers

Los conteos de ratings/reseñas y páginas pueden tener colas largas. El pipeline conserva las columnas originales y agrega versiones capadas al p99 solo para features derivadas de modelado.

In [ ]:
num_cols = ['average_rating', 'ratings_count', 'text_reviews_count', 'num_pages', 'publication_year']
display(books[num_cols].corr(numeric_only=True))
sns.heatmap(books[num_cols].corr(numeric_only=True), annot=True, cmap='vlag', center=0)
plt.title('Correlaciones de metadatos')
plt.show()
display(iqr_outlier_summary(books, num_cols))

## Duplicados

La curación deduplica libros por `book_id`. Las interacciones se deduplican por registro exacto y luego por `review_id`, conservando la versión con `date_updated` más reciente.

In [ ]:
display(duplicate_summary(books_raw, interactions_raw))